# Bài tập lập trình 1 - Phần 2: Chỉ mục nén và Điểm cộngTrong phần này, bạn sẽ xây dựng chỉ mục nén và hiện thực các phương pháp nén bổ sung. Bài tập này được thực hiện **cá nhân**.**Lưu ý:** Phần này yêu cầu bạn đã hoàn thành Phần 1 (chỉ mục không nén). Các lớp và hàm từ Phần 1 (`IdMap`, `UncompressedPostings`, `InvertedIndex`, `BSBIIndex`, `InvertedIndexWriter`, `InvertedIndexIterator`, `InvertedIndexMapper`, `sorted_intersect`, `retrieve`, v.v.) sẽ được sử dụng lại trong phần này.Cụ thể, phần 2 bao gồm các nhiệm vụ sau:1. **Xây dựng chỉ mục nén (50%)** trên cùng tập dữ liệu sử dụng mã hóa độ dài biến đổi (variable length encoding) và hiện thực truy vấn kết hợp Boolean.2. **Viết báo cáo (20%)** phân tích kích thước chỉ mục nén.3. **Điểm cộng (30%)** hiện thực các thuật toán nén khác (ví dụ: gamma-encoding).


## Hướng dẫn nộp bài

1\. Bài tập phải được nộp trước hạn do giảng viên quy định.

2\. Notebook sẽ tự động tạo ra các **file python** trong thư mục submission.

3\. Khi làm bài, **KHÔNG** thay đổi tên class và tên phương thức, các bài kiểm tra tự động sẽ bị lỗi nếu thay đổi.

4\. Bạn cần tải lên **phiên bản PDF** của notebook (chủ yếu dùng để chấm phần báo cáo). File PDF đặt tên theo **mã sinh viên** của bạn (ví dụ: `B22DCKH001.pdf`). Lưu ý rằng việc chuyển đổi PDF trực tiếp sẽ cắt bớt các ô code. Để có phiên bản PDF sử dụng được, trước tiên nhấp vào File > Print Preview, nó sẽ mở trong tab mới, sau đó in thành PDF bằng chức năng in của trình duyệt.

5\. Nộp file PDF báo cáo tại: [Google Drive nộp bài](https://drive.google.com/drive/folders/1jLkXkBEpbIa53FERm5bpN8X9yK8n9bFU?usp=sharing)

6\. Bài tập được thực hiện **cá nhân**. Mọi hành vi sao chép sẽ bị xử lý theo quy định.

## Thiết lập

In [2]:
#Tải magic tee để lưu bản sao của ô khi thực thi
%reload_ext autograding_magics

Thư mục `submission` sẽ chứa tất cả các file cần nộp.

In [3]:
import os
try:
    os.mkdir('submission')
except FileExistsError:
    pass

In [4]:
%%tee submission/imports.py

# Bạn có thể thêm các import bổ sung ở đây
import sys
import pickle as pkl
import array
import os
import timeit
import contextlib

Writing submission/imports.py


# Tập dữ liệu (Corpus)

Tập dữ liệu mà bạn sẽ làm việc trong bài tập này chứa các trang web từ miền stanford.edu. Dữ liệu cho bài tập này có sẵn dưới dạng file .zip tại: http://web.stanford.edu/class/cs276/pa/pa1-data.zip. Đoạn code sau sẽ đặt thư mục dữ liệu vào thư mục hiện tại. Tổng kích thước tập dữ liệu khoảng 170MB.

In [5]:
import urllib.request
import zipfile

data_url = 'http://web.stanford.edu/class/cs276/pa/pa1-data.zip'
data_dir = 'pa1-data'
urllib.request.urlretrieve(data_url, data_dir+'.zip')
zip_ref = zipfile.ZipFile(data_dir+'.zip', 'r')
zip_ref.extractall()
zip_ref.close()

Chỉ mục sẽ được lưu trong `output_dir` và `tmp` sẽ chứa một số file tạm thời cho các chỉ mục thử nghiệm.

In [6]:
try:
    os.mkdir('output_dir')
except FileExistsError:
    pass
try:
    os.mkdir('tmp')
except FileExistsError:
    pass
try:
    os.mkdir('toy_output_dir')
except FileExistsError:
    pass

Có 10 thư mục con (đặt tên từ 0-9) trong thư mục dữ liệu.

In [7]:
sorted(os.listdir('pa1-data'))

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

Mỗi file trong thư mục con là nội dung của một trang web riêng lẻ. Bạn nên giả định rằng tên file là duy nhất trong mỗi thư mục con, nhưng không nhất thiết duy nhất giữa các thư mục con (*nghĩa là* đường dẫn đầy đủ của các file là duy nhất).

In [8]:
sorted(os.listdir('pa1-data/0'))[:10]

['3dradiology.stanford.edu_',
 '3dradiology.stanford.edu_patient_care_Case%2520studies_AVM.html',
 '3dradiology.stanford.edu_patient_care_case_studies.html',
 '5-sure.stanford.edu_',
 '50years.stanford.edu_',
 'a3cservices.stanford.edu_awards_nominate_',
 'a3cservices.stanford.edu_facilities_',
 'a3cservices.stanford.edu_lead_',
 'aa.stanford.edu_',
 'aa.stanford.edu_about_aviation.php']

Tất cả thông tin HTML đã được loại bỏ khỏi các trang web, nên chúng chỉ chứa các từ phân cách bằng dấu cách. Bạn không nên tách từ (tokenize) lại. Mỗi chuỗi liên tiếp các ký tự không phải dấu cách tạo thành một token từ trong tập dữ liệu.

In [9]:
with open('pa1-data/0/3dradiology.stanford.edu_', 'r') as f:
    print(f.read())

3d radiology lab stanford university school of medicine stanford school of medicine 3d and quantitative imaging in the department of radiology search this site only stanford medical sites ways to give find a person alumni lane library ways to give find a person about us mission to develop and apply innovative techniques for efficient quantitative analysis and display of medical imaging data through interdisciplinary collaboration goals education to train physicians and technologists locally and worldwide in the latest developments in 3d and quantitative imaging research to develop new approaches to the exploration analysis and quantitative assesment of diagnostic images that result in a new and or more cost effective diagnostic approaches and b new techniques for the design and planning and monitoring of therapy patient care to deliver valid clinically relevant visualization and analysis of medical imaging data to the stanford community locations richard m lucas magnetic resonance imag

Bạn cũng sẽ tìm thấy một tập dữ liệu nhỏ đi kèm bài tập trong thư mục `toy-data`. Chúng ta sẽ sử dụng tập dữ liệu thử nghiệm này để kiểm tra code trước khi chạy trên tập dữ liệu đầy đủ.

In [10]:
toy_dir = 'toy-data'

# Code cơ sở từ Phần 1Các ô dưới đây chứa code cơ sở từ Phần 1 (chỉ mục không nén). Bạn cần chạy các ô này trước khi thực hiện phần chỉ mục nén.**Lưu ý:** Nếu bạn đã hoàn thành Phần 1, hãy copy các file `.py` từ thư mục `submission` của Phần 1 vào đây, hoặc chạy lại các ô code dưới đây với code đã hoàn thành.

## IdMap

Trích dẫn từ Mục 4.2:

> Để việc xây dựng chỉ mục hiệu quả hơn, chúng ta biểu diễn các term dưới dạng termID (thay vì chuỗi ký tự), trong đó mỗi termID là một số thứ tự duy nhất. Chúng ta có thể xây dựng ánh xạ từ term sang termID trong quá trình xử lý bộ sưu tập. Tương tự, chúng ta cũng biểu diễn tài liệu dưới dạng docID (thay vì chuỗi ký tự).

Trước tiên, hãy xây dựng lớp trợ giúp `IdMap`, ánh xạ chuỗi sang id số (và ngược lại). Chúng ta sẽ cần điều này để ánh xạ song ánh term sang termID và doc sang docID.

Hãy điền vào các hàm `_get_str` và `_get_id` trong đoạn code sau. Giao diện duy nhất với lớp này được cung cấp bởi `__getitem__`, hàm này lấy ánh xạ chính xác tùy thuộc vào kiểu của key. (Xem [tài liệu này](https://docs.python.org/3.7/reference/datamodel.html#special-method-names) nếu bạn chưa quen với các hàm đặc biệt như `__getitem__`). Ngoài ra, để đơn giản hóa code sau này, chúng ta cũng sẽ thêm logic để thêm chuỗi nếu nó chưa tồn tại trong map.

Chúng ta sẽ sử dụng dictionary để truy cập hiệu quả từ chuỗi sang id số, và sử dụng list để truy cập (và lưu trữ) hiệu quả từ id số sang chuỗi.

Bạn cũng có thể thấy [hướng dẫn ngắn này](https://www.omkarpathak.in/2018/04/11/python-getitem-and-setitem/) hữu ích để bắt đầu với các phương thức đặc biệt (hay còn gọi là magic methods)

In [11]:
%%tee submission/id_map.py
class IdMap:
    """Lớp trợ giúp để lưu trữ ánh xạ từ chuỗi sang id."""
    def __init__(self):
        self.str_to_id = {}
        self.id_to_str = []

    def __len__(self):
        """Trả về số lượng term được lưu trong IdMap"""
        return len(self.id_to_str)

    def _get_str(self, i):
        """Trả về chuỗi tương ứng với id (`i`) cho trước."""
        ### Bắt đầu code của bạn
        if 0 <= i < len(self.id_to_str):
            return self.id_to_str[i]
        raise IndexError("ID không tồn tại trong map")
        ### Kết thúc code của bạn

    def _get_id(self, s):
        """Trả về id tương ứng với chuỗi (`s`).
        Nếu `s` chưa có trong IdMap, gán một id mới và trả về id mới đó.
        """
        ### Bắt đầu code của bạn
        if s not in self.str_to_id:
            new_id = len(self.id_to_str)
            self.str_to_id[s] = new_id
            self.id_to_str.append(s)
        return self.str_to_id[s]
        ### Kết thúc code của bạn

    def __getitem__(self, key):
        """Nếu `key` là số nguyên, sử dụng _get_str;
           Nếu `key` là chuỗi, sử dụng _get_id;"""
        if type(key) is int:
            return self._get_str(key)
        elif type(key) is str:
            return self._get_id(key)
        else:
            raise TypeError

Writing submission/id_map.py


## Mã hóa danh sách Posting dưới dạng bytearray

Để ghi và đọc danh sách posting (docID) hiệu quả từ đĩa, chúng ta lưu chúng dưới dạng bytearray. Chúng tôi đã cung cấp lớp `UncompressedPostings` chứa các hàm encode và decode tĩnh. Trong phần tiếp theo, bạn sẽ phải hiện thực phiên bản nén với cùng giao diện.

Tham khảo:
1. https://docs.python.org/3/library/array.html
2. https://pymotw.com/3/array/#module-array

In [12]:
class UncompressedPostings:

    @staticmethod
    def encode(postings_list):
        """Mã hóa postings_list thành luồng byte

        Tham số
        ----------
        postings_list: List[int]
            Danh sách các docID (posting)

        Trả về
        -------
        bytes
            bytearray biểu diễn các số nguyên trong postings_list
        """
        return array.array('L', postings_list).tobytes()

    @staticmethod
    def decode(encoded_postings_list):
        """Giải mã postings_list từ luồng byte

        Tham số
        ----------
        encoded_postings_list: bytes
            bytearray biểu diễn danh sách posting đã mã hóa như đầu ra
            của hàm encode

        Trả về
        -------
        List[int]
            Danh sách docID đã giải mã từ encoded_postings_list
        """

        decoded_postings_list = array.array('L')
        decoded_postings_list.frombytes(encoded_postings_list)
        return decoded_postings_list.tolist()

## Chỉ mục đảo trên Đĩa

> Khi bộ nhớ chính không đủ, chúng ta cần sử dụng thuật toán sắp xếp ngoại (external sorting), tức là thuật toán sử dụng đĩa. Để đạt tốc độ chấp nhận được, yêu cầu chính của thuật toán này là giảm thiểu số lần tìm kiếm ngẫu nhiên trên đĩa trong quá trình sắp xếp - đọc tuần tự trên đĩa nhanh hơn nhiều so với tìm kiếm.

Trong phần này, chúng tôi cung cấp lớp cơ sở `InvertedIndex`, lớp này sau đó sẽ được kế thừa thành `InvertedIndexWriter`, `InvertedIndexIterator` và `InvertedIndexMapper`. Mặc dù thông thường `cPickle` được sử dụng để tuần tự hóa trong Python, nó không hỗ trợ đọc và ghi một phần, đó là lý do chúng ta tự xây dựng giải pháp tùy chỉnh.

In [13]:
class InvertedIndex:
    """Lớp hiện thực đọc và ghi hiệu quả chỉ mục đảo lên đĩa

    Thuộc tính
    ----------
    postings_dict: Dictionary ánh xạ: termID->(vị_trí_bắt_đầu_trong_file_index,
                                                số_posting_trong_danh_sách,
                                               độ_dài_byte_của_danh_sách_posting)
        Đây là dictionary ánh xạ từ termID sang bộ 3 metadata
        hữu ích cho việc đọc và ghi posting trong file chỉ mục
        lên/từ đĩa. Ánh xạ này được giữ trong bộ nhớ.
        vị_trí_bắt_đầu_trong_file_index là vị trí (tính bằng byte) của danh sách
        posting trong file chỉ mục
        số_posting_trong_danh_sách là số lượng posting (docID) trong
        danh sách posting
        độ_dài_byte_của_danh_sách_posting là độ dài mã hóa byte
        của danh sách posting

    terms: List[int]
        Danh sách các termID để ghi nhớ thứ tự mà các term và danh sách
        posting của chúng được thêm vào chỉ mục.

        Sau Python 3.7, về mặt kỹ thuật chúng ta không cần nó nữa vì dict
        trong Python là OrderedDict, nhưng vì đây là tính năng tương đối mới,
        chúng ta vẫn duy trì tương thích ngược bằng list để theo dõi thứ tự
        chèn.
    """
    def __init__(self, index_name, postings_encoding=None, directory=''):
        """
        Tham số
        ----------
        index_name (str): Tên dùng để lưu các file liên quan đến chỉ mục
        postings_encoding: Lớp hiện thực các phương thức tĩnh để mã hóa và
            giải mã danh sách số nguyên. Mặc định là None, sẽ được thay thế
            bằng UncompressedPostings
        directory (str): Thư mục nơi các file chỉ mục sẽ được lưu
        """

        self.index_file_path = os.path.join(directory, index_name+'.index')
        self.metadata_file_path = os.path.join(directory, index_name+'.dict')

        if postings_encoding is None:
            self.postings_encoding = UncompressedPostings
        else:
            self.postings_encoding = postings_encoding
        self.directory = directory

        self.postings_dict = {}
        self.terms = []         #Cần theo dõi thứ tự chèn các term.
                                #Không cần thiết từ Python 3.7 trở đi

    def __enter__(self):
        """Mở index_file và tải metadata khi vào context"""
        # Mở file chỉ mục
        self.index_file = open(self.index_file_path, 'rb+')

        # Tải postings dict và terms từ file metadata
        with open(self.metadata_file_path, 'rb') as f:
            self.postings_dict, self.terms = pkl.load(f)
            self.term_iter = self.terms.__iter__()

        return self

    def __exit__(self, exception_type, exception_value, traceback):
        """Đóng index_file và lưu metadata khi thoát context"""
        # Đóng file chỉ mục
        self.index_file.close()

        # Ghi postings dict và terms vào file metadata
        with open(self.metadata_file_path, 'wb') as f:
            pkl.dump([self.postings_dict, self.terms], f)

Vì chúng ta đang tương tác với file trên đĩa, chúng tôi cung cấp các hàm `__enter__` và `__exit__`, cho phép sử dụng câu lệnh `with` giống như khi làm việc với file IO trong Python (Tham khảo [tài liệu chính thức về context manager](https://docs.python.org/3/library/contextlib.html)).

Đây là ví dụ cách sử dụng context manager `InvertedIndexWriter`:

```python
with InvertedIndexWriter('test', directory='tmp/') as index:
    # Code ở đây
```

Nếu bạn chưa quen với context manager trong Python, chúng tôi khuyên bạn nên xem các hướng dẫn này ([1](https://jeffknupp.com/blog/2016/03/07/python-with-context-managers/), [2](http://arnavk.com/posts/python-context-managers/)), mặc dù không nhất thiết phải hiểu tất cả.

## Đánh chỉ mục

> BSBI (i) phân chia bộ sưu tập thành các phần có kích thước bằng nhau, (ii) sắp xếp các cặp termID-docID của mỗi phần trong bộ nhớ, (iii) lưu các kết quả trung gian đã sắp xếp lên đĩa, và (iv) hợp nhất tất cả kết quả trung gian thành chỉ mục cuối cùng.

Bạn nên coi mỗi thư mục con là một khối (block), và chỉ tải một khối vào bộ nhớ tại một thời điểm khi xây dựng chỉ mục (lưu ý: bạn sẽ bị trừ điểm nếu xây dựng chỉ mục bằng cách tải các khối lớn hơn một thư mục vào bộ nhớ cùng lúc). Lưu ý rằng chúng ta đang trừu tượng hóa khái niệm block của hệ điều hành. Bạn có thể giả định rằng mỗi khối đủ nhỏ để lưu trong bộ nhớ.

Trong phần này, chúng tôi giới thiệu lớp `BSBIIndex` mà chúng ta sẽ xây dựng dần dần. Hàm `index` cung cấp khung cho BSBI và nhiệm vụ của bạn là hiện thực các hàm `parse_block`, `invert_write` và `merge` trong các phần tiếp theo.

In [15]:

# Không thay đổi gì ở đây, chúng sẽ bị ghi đè khi chấm điểm
class BSBIIndex:
    """
    Thuộc tính
    ----------
    term_id_map(IdMap): Để ánh xạ term sang termID
    doc_id_map(IdMap): Để ánh xạ đường dẫn tương đối của tài liệu (ví dụ
        0/3dradiology.stanford.edu_) sang docID
    data_dir(str): Đường dẫn đến dữ liệu
    output_dir(str): Đường dẫn đến thư mục chứa file chỉ mục đầu ra
    index_name(str): Tên gán cho chỉ mục
    postings_encoding: Mã hóa dùng để lưu posting.
        Mặc định (None) ngầm định sử dụng UncompressedPostings
    """
    def __init__(self, data_dir, output_dir, index_name = "BSBI",
                 postings_encoding = None):
        self.term_id_map = IdMap()
        self.doc_id_map = IdMap()
        self.data_dir = data_dir
        self.output_dir = output_dir
        self.index_name = index_name
        self.postings_encoding = postings_encoding

        # Lưu tên các chỉ mục trung gian
        self.intermediate_indices = []

    def save(self):
        """Lưu doc_id_map và term_id_map vào thư mục đầu ra"""

        with open(os.path.join(self.output_dir, 'terms.dict'), 'wb') as f:
            pkl.dump(self.term_id_map, f)
        with open(os.path.join(self.output_dir, 'docs.dict'), 'wb') as f:
            pkl.dump(self.doc_id_map, f)

    def load(self):
        """Tải doc_id_map và term_id_map từ thư mục đầu ra"""

        with open(os.path.join(self.output_dir, 'terms.dict'), 'rb') as f:
            self.term_id_map = pkl.load(f)
        with open(os.path.join(self.output_dir, 'docs.dict'), 'rb') as f:
            self.doc_id_map = pkl.load(f)

    def index(self):
        """Code đánh chỉ mục cơ sở

        Hàm này duyệt qua các thư mục dữ liệu,
        gọi parse_block để phân tích tài liệu
        gọi invert_write để đảo ngược mỗi khối và ghi vào chỉ mục mới
        sau đó lưu các id map và gọi merge trên các chỉ mục trung gian
        """
        for block_dir_relative in sorted(next(os.walk(self.data_dir))[1]):
            td_pairs = self.parse_block(block_dir_relative)
            index_id = 'index_'+block_dir_relative
            self.intermediate_indices.append(index_id)
            with InvertedIndexWriter(index_id, directory=self.output_dir,
                                     postings_encoding=
                                     self.postings_encoding) as index:
                self.invert_write(td_pairs, index)
                td_pairs = None
        self.save()
        with InvertedIndexWriter(self.index_name, directory=self.output_dir,
                                 postings_encoding=
                                 self.postings_encoding) as merged_index:
            with contextlib.ExitStack() as stack:
                indices = [stack.enter_context(
                    InvertedIndexIterator(index_id,
                                          directory=self.output_dir,
                                          postings_encoding=
                                          self.postings_encoding))
                 for index_id in self.intermediate_indices]
                self.merge(indices, merged_index)

### Phân tích (Parsing)

> Hàm `parse_block` phân tích tài liệu thành các cặp termID-docID và tích lũy các cặp trong bộ nhớ cho đến khi một khối có kích thước cố định đầy. Chúng ta chọn kích thước khối vừa vặn trong bộ nhớ để cho phép sắp xếp nhanh trong bộ nhớ.

Bạn nên coi mỗi thư mục con là một khối; `parse_block` nhận đường dẫn đến thư mục con (khối). Bạn nên giả định rằng tên file là duy nhất trong mỗi thư mục con, nhưng không nhất thiết duy nhất giữa các thư mục con (nghĩa là đường dẫn đầy đủ của file là duy nhất).

_Lưu ý - Chúng ta đang kế thừa `BSBIIndex` từ `BSBIIndex`, đây chỉ là cách đơn giản để thêm phương thức mới vào lớp đã tồn tại. Xin đừng nhầm lẫn, nó chỉ được sử dụng như công cụ giảng dạy để chia định nghĩa lớp trong jupyter notebook và không có phép thuật nào khác ở đây._

In [16]:
%%tee submission/parse_block.py
class BSBIIndex(BSBIIndex):
    def parse_block(self, block_dir_relative):
        """Phân tích file văn bản đã tokenize thành các cặp termID-docID

        Tham số
        ----------
        block_dir_relative : str
            Đường dẫn tương đối đến thư mục chứa các file của khối

        Trả về
        -------
        List[Tuple[Int, Int]]
            Trả về tất cả các cặp td_pairs được trích xuất từ khối

        Nên sử dụng self.term_id_map và self.doc_id_map để lấy termID và docID.
        Chúng được duy trì qua các lần gọi parse_block
        """
        ### Bắt đầu code của bạn
        td_pairs = []
        block_path = os.path.join(self.data_dir, block_dir_relative)

        # Đọc tất cả các file trong thư mục block
        for filename in os.listdir(block_path):
            # Tạo đường dẫn tương đối (ví dụ: '0/3dradiology.stanford.edu_')
            doc_path_relative = os.path.join(block_dir_relative, filename).replace('\\', '/')
            doc_id = self.doc_id_map[doc_path_relative]

            file_path = os.path.join(block_path, filename)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                tokens = content.split()
                for token in tokens:
                    term_id = self.term_id_map[token]
                    td_pairs.append((term_id, doc_id))

        return td_pairs
        ### Kết thúc code của bạn

Overwriting submission/parse_block.py


### Đảo ngược (Inversion)

> Khối sau đó được đảo ngược và ghi lên đĩa. Đảo ngược bao gồm hai bước. Đầu tiên, chúng ta sắp xếp các cặp termID-docID. Tiếp theo, chúng ta tập hợp tất cả các cặp termID-docID có cùng termID thành một danh sách posting, trong đó một posting đơn giản là một docID. Kết quả, một chỉ mục đảo cho khối vừa đọc, sau đó được ghi lên đĩa.

Trong phần này, chúng ta thêm hàm `invert_write` thực hiện chính xác điều này.

Tuy nhiên, trước tiên chúng ta cần hiện thực lớp `InvertedIndexWriter`. Lớp này cung cấp hàm append, giống như list, nhưng postings_list không được lưu trong bộ nhớ mà được ghi trực tiếp lên đĩa.

In [17]:
%%tee submission/inverted_index_writer.py
class InvertedIndexWriter(InvertedIndex):
    """"""
    def __enter__(self):
        self.index_file = open(self.index_file_path, 'wb+')
        return self

    def append(self, term, postings_list):
        """Thêm term và postings_list vào cuối file chỉ mục.

        Hàm này thực hiện ba việc:
        1. Mã hóa postings_list sử dụng self.postings_encoding
        2. Lưu metadata dưới dạng self.terms và self.postings_dict
           Lưu ý rằng self.postings_dict ánh xạ termID sang bộ 3 gồm
           (vị_trí_bắt_đầu_trong_file_index,
           số_posting_trong_danh_sách,
           độ_dài_byte_của_danh_sách_posting)
        3. Thêm luồng byte vào file chỉ mục trên đĩa

        Gợi ý: Bạn có thể thấy hữu ích khi đọc tài liệu Python I/O
        (https://docs.python.org/3/tutorial/inputoutput.html) để biết
        thông tin về cách thêm vào cuối file.

        Tham số
        ----------
        term:
            term hoặc termID là định danh duy nhất cho term
        postings_list: List[Int]
            Danh sách các docID nơi term xuất hiện
        """
        ### Bắt đầu code của bạn
        encoded_postings = self.postings_encoding.encode(postings_list)
        start_pos = self.index_file.tell()
        self.index_file.write(encoded_postings)

        self.terms.append(term)
        self.postings_dict[term] = (start_pos, len(postings_list), len(encoded_postings))
        ### Kết thúc code của bạn

Writing submission/inverted_index_writer.py


Bây giờ chúng ta hiện thực `invert_write`, nhận vào td_pairs được tạo bởi parse_block. Chúng ta sử dụng `InvertedIndexWriter` để ghi chúng lên đĩa.

In [18]:
%%tee submission/invert_write.py
class BSBIIndex(BSBIIndex):
    def invert_write(self, td_pairs, index):
        """Đảo ngược td_pairs thành danh sách posting và ghi chúng vào chỉ mục cho trước

        Tham số
        ----------
        td_pairs: List[Tuple[Int, Int]]
            Danh sách các cặp termID-docID
        index: InvertedIndexWriter
            Chỉ mục đảo trên đĩa tương ứng với khối
        """
        ### Bắt đầu code của bạn
        from collections import defaultdict

        # Nhóm docID theo termID
        term_dict = defaultdict(list)
        for term_id, doc_id in td_pairs:
            term_dict[term_id].append(doc_id)

        # Sắp xếp các termID để ghi tuần tự
        for term_id in sorted(term_dict.keys()):
            # Sắp xếp và loại bỏ docID trùng lặp cho mỗi term
            postings_list = sorted(list(set(term_dict[term_id])))
            index.append(term_id, postings_list)
        ### Kết thúc code của bạn

Writing submission/invert_write.py


### Hợp nhất (Merging)

> Thuật toán đồng thời hợp nhất mười khối thành một chỉ mục lớn đã hợp nhất. Để thực hiện hợp nhất, chúng ta mở tất cả file khối đồng thời, và duy trì các bộ đệm đọc nhỏ cho mười khối đang đọc và một bộ đệm ghi cho chỉ mục hợp nhất cuối cùng đang ghi.

Mô hình iterator trong Python tự nhiên phù hợp với nhu cầu duy trì bộ đệm đọc nhỏ. Chúng ta có thể duyệt qua file trong khi chỉ đọc một danh sách posting tại một thời điểm từ đĩa. Chúng ta kế thừa `InvertedIndex` thành `InvertedIndexIterator` để thực hiện phần duyệt.

In [19]:
%%tee submission/inverted_index_iterator.py
class InvertedIndexIterator(InvertedIndex):
    """"""
    def __enter__(self):
        """Thêm initialization_hook vào hàm __enter__ của lớp cha
        """
        super().__enter__()
        self._initialization_hook()
        return self

    def _initialization_hook(self):
        """Sử dụng hàm này để khởi tạo iterator
        """
        ### Bắt đầu code của bạn
        self.term_iter = iter(self.terms)
        ### Kết thúc code của bạn

    def __iter__(self):
        return self

    def __next__(self):
        """Trả về cặp (term, postings_list) tiếp theo trong chỉ mục.

        Lưu ý: Hàm này chỉ nên đọc một lượng nhỏ dữ liệu từ
        file chỉ mục. Cụ thể, bạn không nên cố gắng giữ toàn bộ
        file chỉ mục trong bộ nhớ.
        """
        ### Bắt đầu code của bạn
        try:
            term = next(self.term_iter)
        except StopIteration:
            raise StopIteration

        start_pos, _, byte_length = self.postings_dict[term]
        self.index_file.seek(start_pos)
        encoded_postings = self.index_file.read(byte_length)
        postings_list = self.postings_encoding.decode(encoded_postings)

        return term, postings_list
        ### Kết thúc code của bạn

    def delete_from_disk(self):
        """Đánh dấu chỉ mục để xóa khi thoát. Hữu ích cho chỉ mục tạm thời
        """
        self.delete_upon_exit = True

    def __exit__(self, exception_type, exception_value, traceback):
        """Xóa file chỉ mục khi thoát context cùng với
        các hàm __exit__ của lớp cha"""
        self.index_file.close()
        if hasattr(self, 'delete_upon_exit') and self.delete_upon_exit:
            os.remove(self.index_file_path)
            os.remove(self.metadata_file_path)
        else:
            with open(self.metadata_file_path, 'wb') as f:
                pkl.dump([self.postings_dict, self.terms], f)

Writing submission/inverted_index_iterator.py


> Trong quá trình hợp nhất, ở mỗi lần lặp, chúng ta chọn termID thấp nhất chưa được xử lý bằng hàng đợi ưu tiên hoặc cấu trúc dữ liệu tương tự. Tất cả danh sách posting cho termID đó được đọc và hợp nhất, và danh sách đã hợp nhất được ghi lại lên đĩa. Mỗi bộ đệm đọc được nạp lại từ file khi cần thiết.

Chúng ta sẽ sử dụng `InvertedIndexIterator` để đọc và `InvertedIndexWriter` để ghi danh sách posting đã hợp nhất.

Lưu ý rằng chúng ta đã mở mỗi chỉ mục đảo bằng câu lệnh `with`, nhưng bây giờ cần làm điều đó theo chương trình. Bạn có thể thấy chúng tôi đã sử dụng `contextlib` ([tài liệu](https://docs.python.org/3/library/contextlib.html#contextlib.ExitStack)) trong hàm `index` đã cung cấp của lớp `BSBIIndex` cơ sở. Nhiệm vụ của bạn là **viết logic** để hợp nhất các đối tượng `InvertedIndexIterator` đã mở và ghi từng danh sách posting vào một đối tượng `InvertedIndexWriter` duy nhất.

Vì chúng ta biết các posting đã được sắp xếp, chúng ta có thể hợp nhất chúng theo thứ tự trong thời gian tuyến tính. Thực tế `heapq` ([tài liệu](https://docs.python.org/3.0/library/heapq.html)) là module chuẩn Python cung cấp hiện thực thuật toán hàng đợi heap. Cụ thể, nó chứa hàm tiện ích `heapq.merge` hợp nhất nhiều đầu vào đã sắp xếp thành một đầu ra đã sắp xếp và trả về iterator trên các giá trị đã sắp xếp. Không chỉ hữu ích cho hợp nhất danh sách posting, mà còn cho hợp nhất các chỉ mục đảo.

Để giúp bạn bắt đầu với hàm `heapq.merge`, chúng tôi cung cấp ví dụ sử dụng. Hai danh sách chứa động vật/chim được sắp xếp theo tuổi thọ trung bình. Chúng ta muốn hợp nhất hai danh sách.

In [20]:
import heapq
animal_lifespans = [('Giraffe', 28),
                   ('Rhinoceros', 40),
                   ('Indian Elephant', 70),
                   ('Golden Eagle', 80),
                   ('Box turtle', 123)]

tree_lifespans = [('Gray Birch', 50),
                  ('Black Willow', 70),
                  ('Basswood', 100),
                  ('Bald Cypress', 600)]

lifespan_lists = [animal_lifespans, tree_lifespans]

for merged_item in heapq.merge(*lifespan_lists, key=lambda x: x[1]):
    print(merged_item)

('Giraffe', 28)
('Rhinoceros', 40)
('Gray Birch', 50)
('Indian Elephant', 70)
('Black Willow', 70)
('Golden Eagle', 80)
('Basswood', 100)
('Box turtle', 123)
('Bald Cypress', 600)


Lưu ý cách sử dụng `*` để giải nén `lifespan_lists` thành đối số và hàm `lambda` để biểu diễn khóa sắp xếp. Nếu chưa quen, bạn có thể xem tài liệu ([giải nén danh sách](https://docs.python.org/3/tutorial/controlflow.html#unpacking-argument-lists), [biểu thức lambda](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions)) hoặc hướng dẫn ([giải nén danh sách](https://www.geeksforgeeks.org/packing-and-unpacking-arguments-in-python/), [biểu thức lambda](https://www.afternerd.com/blog/python-lambdas/))

In [21]:
%%tee submission/merge.py

import heapq
class BSBIIndex(BSBIIndex):
    def merge(self, indices, merged_index):
        """Hợp nhất nhiều chỉ mục đảo thành một chỉ mục duy nhất

        Tham số
        ----------
        indices: List[InvertedIndexIterator]
            Danh sách các đối tượng InvertedIndexIterator, mỗi đối tượng
            đại diện cho một chỉ mục đảo có thể duyệt cho một khối
        merged_index: InvertedIndexWriter
            Một đối tượng InvertedIndexWriter để ghi từng danh sách
            posting đã hợp nhất
        """
        ### Bắt đầu code của bạn
        # Sử dụng heapq.merge để hợp nhất các iterator, ưu tiên term_id (x[0])
        merged_iter = heapq.merge(*indices, key=lambda x: x[0])

        current_term = None
        current_postings = []

        for term, postings in merged_iter:
            if term != current_term:
                if current_term is not None:
                    # Ghi term trước đó với postings đã hợp nhất, sắp xếp, loại trùng
                    merged_postings = sorted(list(set(current_postings)))
                    merged_index.append(current_term, merged_postings)
                current_term = term
                current_postings = postings
            else:
                current_postings.extend(postings)

        # Đảm bảo ghi term cuối cùng
        if current_term is not None:
            merged_postings = sorted(list(set(current_postings)))
            merged_index.append(current_term, merged_postings)
        ### Kết thúc code của bạn

Writing submission/merge.py


## Truy xuất kết hợp Boolean (Boolean conjunctive retrieval)

Chúng ta sẽ hiện thực hàm `retrieve` cho BSBIIndex, hàm này nhận một chuỗi truy vấn gồm các token phân cách bằng dấu cách, trả về danh sách tài liệu chứa mỗi token trong truy vấn. Tuy nhiên, chúng ta không muốn duyệt qua chỉ mục hay tải toàn bộ chỉ mục để tìm các term liên quan.

Trước tiên, chúng ta sẽ hiện thực `InvertedIndexMapper` kế thừa `InvertedIndex` để thêm chức năng truy xuất posting tương ứng với một term cụ thể bằng cách seek đến vị trí đó trong file.

In [22]:
%%tee submission/inverted_index_mapper.py
class InvertedIndexMapper(InvertedIndex):
    def __getitem__(self, key):
        return self._get_postings_list(key)

    def _get_postings_list(self, term):
        """Lấy danh sách posting (các docId) cho `term`.

        Hàm này không nên duyệt qua file chỉ mục.
        Tức là, nó chỉ cần đọc các byte từ file chỉ mục
        tương ứng với danh sách posting cho term được yêu cầu.
        """
        ### Bắt đầu code của bạn
        if term not in self.postings_dict:
            return []

        start_pos, _, byte_length = self.postings_dict[term]
        self.index_file.seek(start_pos)
        encoded_postings = self.index_file.read(byte_length)

        return self.postings_encoding.decode(encoded_postings)
        ### Kết thúc code của bạn

Writing submission/inverted_index_mapper.py


Bây giờ chúng ta có thể lấy danh sách posting tương ứng với các term truy vấn, chúng ta muốn giao chúng. Tuy nhiên, tương tự như hợp nhất, chúng ta có thể sử dụng tính chất đã sắp xếp của các danh sách này để giao chúng rất hiệu quả. Ở đây chúng ta sẽ hiện thực `sorted_intersect` nhận hai danh sách đã sắp xếp và trả về giao đã sắp xếp của các phần tử.

In [23]:
%%tee submission/sorted_intersect.py
def sorted_intersect(list1, list2):
    """Giao hai danh sách đã sắp xếp (tăng dần) và trả về kết quả đã sắp xếp

    Tham số
    ----------
    list1: List[Comparable]
    list2: List[Comparable]
        Các danh sách đã sắp xếp cần giao

    Trả về
    -------
    List[Comparable]
        Giao đã sắp xếp
    """
    ### Bắt đầu code của bạn
    result = []
    i, j = 0, 0
    while i < len(list1) and j < len(list2):
        if list1[i] == list2[j]:
            result.append(list1[i])
            i += 1
            j += 1
        elif list1[i] < list2[j]:
            i += 1
        else:
            j += 1
    return result
    ### Kết thúc code của bạn

Writing submission/sorted_intersect.py


Bây giờ chúng ta sẵn sàng viết hàm `retrieve` sử dụng `InvertedIndexMapper` và `sorted_intersect`

In [24]:
%%tee submission/retrieve.py
class BSBIIndex(BSBIIndex):
    def retrieve(self, query):
        """Truy xuất các tài liệu tương ứng với truy vấn kết hợp

        Tham số
        ----------
        query: str
            Danh sách các token truy vấn phân cách bằng dấu cách

        Kết quả
        ------
        List[str]
            Danh sách tài liệu đã sắp xếp chứa mỗi token truy vấn.
            Trả về rỗng nếu không tìm thấy tài liệu.

        KHÔNG nên ném lỗi cho các term không có trong tập dữ liệu
        """
        if len(self.term_id_map) == 0 or len(self.doc_id_map) == 0:
            self.load()

        ### Bắt đầu code của bạn
        tokens = query.strip().split()
        if not tokens:
            return []

        # Tìm termID cho các tokens, nếu có token chưa từng xuất hiện, trả về []
        term_ids = []
        for token in tokens:
            if token not in self.term_id_map.str_to_id:
                return []
            term_ids.append(self.term_id_map.str_to_id[token])

        # Mở index để đọc postings
        with InvertedIndexMapper(self.index_name, directory=self.output_dir, postings_encoding=self.postings_encoding) as mapper:
            postings_lists = [mapper[term_id] for term_id in term_ids]

        if not postings_lists:
            return []

        # Tối ưu: Giao các list theo thứ tự từ nhỏ đến lớn sẽ nhanh hơn
        postings_lists.sort(key=len)

        result_docs = postings_lists[0]
        for p in postings_lists[1:]:
            result_docs = sorted_intersect(result_docs, p)
            if not result_docs:
                break

        # Chuyển đổi docID thành đường dẫn file
        result_paths = [self.doc_id_map[doc_id] for doc_id in result_docs]
        return sorted(result_paths)
        ### Kết thúc code của bạn

Writing submission/retrieve.py


# Xây dựng chỉ mục nén (50%)


Trong phần này, bạn nên xây dựng chỉ mục nén sử dụng mã hóa khoảng cách (gap encoding) với mã hóa byte biến đổi (variable byte encoding) cho mỗi khoảng cách.

Đây là một số thông tin hữu ích về Python.

In [25]:
# Toán tử chia dư (modulo) %

print("10 % 2 = ", 10 % 2)
print("10 % 3 = ", 10 % 3)

# Phép chia lấy phần nguyên trong Python 3 dùng hai dấu gạch chéo

print("10 / 3 = ", 10 / 3)
print("10 // 3 = ", 10 // 3)

10 % 2 =  0
10 % 3 =  1
10 / 3 =  3.3333333333333335
10 // 3 =  3


Điền vào lớp `CompressedPostings` sau đây, lớp này sẽ được sử dụng thay thế trực tiếp cho `UncompressedPostings`. Bạn nên tham khảo [Chương 5](https://nlp.stanford.edu/IR-book/pdf/05comp.pdf) của sách giáo khoa để hiểu cách mã hóa khoảng cách với mã hóa byte biến đổi cho mỗi khoảng cách.

In [38]:
%%tee submission/compressed_postings.py
class CompressedPostings:
    #Nếu bạn cần thêm phương thức trợ giúp, thêm ở đây
    ### Bắt đầu code của bạn
    @staticmethod
    def vb_encode_number(n):
        bytes_list = []
        while True:
            bytes_list.insert(0, n % 128)
            if n < 128:
                break
            n = n // 128
        bytes_list[-1] += 128
        return bytes_list
    ### Kết thúc code của bạn

    @staticmethod
    def encode(postings_list):
        """Mã hóa `postings_list` sử dụng mã hóa khoảng cách (gap encoding)
        với mã hóa byte biến đổi cho mỗi khoảng cách

        Tham số
        ----------
        postings_list: List[int]
            Danh sách posting cần mã hóa

        Trả về
        -------
        bytes:
            Biểu diễn byte của danh sách posting đã nén
            (được tạo bởi hàm `array.tobytes`)
        """
        ### Bắt đầu code của bạn
        if not postings_list:
            return b''
        encoded_bytes = []
        last_doc = 0
        for doc in postings_list:
            gap = doc - last_doc
            encoded_bytes.extend(CompressedPostings.vb_encode_number(gap))
            last_doc = doc
        return bytes(encoded_bytes)
        ### Kết thúc code của bạn


    @staticmethod
    def decode(encoded_postings_list):
        """Giải mã biểu diễn byte của danh sách posting đã nén

        Tham số
        ----------
        encoded_postings_list: bytes
            Biểu diễn byte được tạo bởi `CompressedPostings.encode`

        Trả về
        -------
        List[int]
            Danh sách posting đã giải mã (mỗi posting là một docId)
        """
        ### Bắt đầu code của bạn
        postings = []
        last_doc = 0
        n = 0
        for byte in encoded_postings_list:
            if byte < 128:
                n = 128 * n + byte
            else:
                n = 128 * n + (byte - 128)
                last_doc += n
                postings.append(last_doc)
                n = 0
        return postings
        ### Kết thúc code của bạn

Overwriting submission/compressed_postings.py


Trước tiên, viết một số test case cho các phương thức trợ giúp mà bạn có thể đã hiện thực

In [41]:
### Bắt đầu code của bạn
    # Test case 1: Số nhỏ hơn 128 (chỉ cần 1 byte, bit đầu tiên là 1)
    # 5 -> nhị phân: 10000101 -> thập phân: 133
assert CompressedPostings.vb_encode_number(5) == [133], "Lỗi mã hóa số < 128"

    # Test case 2: Số 127 (giá trị lớn nhất biểu diễn được bằng 1 byte trong VB)
    # 127 -> nhị phân: 11111111 -> thập phân: 255
assert CompressedPostings.vb_encode_number(127) == [255], "Lỗi mã hóa số 127"

    # Test case 3: Số 128 (cần 2 byte: byte đầu bit 0, byte cuối bit 1)
    # 128 = 1 * 128 + 0 -> [1, 0 + 128] = [1, 128]
assert CompressedPostings.vb_encode_number(128) == [1, 128], "Lỗi mã hóa số 128"

    # Test case 4: Số lớn (cần 3 byte)
    # 214577 = 13 * 128^2 + 12 * 128^1 + 49
    # -> [13, 12, 49 + 128] = [13, 12, 177]
assert CompressedPostings.vb_encode_number(214577) == [13, 12, 177], "Lỗi mã hóa số lớn"

print("Tất cả test case cho phương thức trợ giúp vb_encode_number đều chạy thành công!")
    ### Kết thúc code của bạn

Tất cả test case cho phương thức trợ giúp vb_encode_number đều chạy thành công!


Chúng tôi đã cung cấp hàm trợ giúp đơn giản để bạn kiểm tra xem danh sách posting đã mã hóa có được giải mã đúng không.

In [42]:
def test_encode_decode(l):
    e = CompressedPostings.encode(l)
    d = CompressedPostings.decode(e)
    assert d == l
    print(l, e)

Viết một vài test case để đảm bảo việc nén và giải nén danh sách posting được thực hiện đúng

In [43]:
### Bắt đầu code của bạn
test_encode_decode([1, 2, 3, 4, 5])
test_encode_decode([10, 20, 30, 40, 100, 200])
test_encode_decode([130, 256, 512, 1024, 2048])
test_encode_decode([824, 829, 215406]) # Một ví dụ gap lớn
### Kết thúc code của bạn

[1, 2, 3, 4, 5] b'\x81\x81\x81\x81\x81'
[10, 20, 30, 40, 100, 200] b'\x8a\x8a\x8a\x8a\xbc\xe4'
[130, 256, 512, 1024, 2048] b'\x01\x82\xfe\x02\x80\x04\x80\x08\x80'
[824, 829, 215406] b'\x06\xb8\x85\r\x0c\xb1'


Bây giờ hãy tạo thư mục đầu ra mới và sử dụng `CompressedPostings` để tạo chỉ mục nén

In [44]:
try:
    os.mkdir('output_dir_compressed')
except FileExistsError:
    pass

In [45]:
BSBI_instance_compressed = BSBIIndex(data_dir='pa1-data', output_dir = 'output_dir_compressed', postings_encoding=CompressedPostings)
BSBI_instance_compressed.index()

In [46]:
BSBI_instance_compressed = BSBIIndex(data_dir='pa1-data', output_dir = 'output_dir_compressed', postings_encoding=CompressedPostings)
BSBI_instance_compressed.retrieve('boolean retrieval')

['1/cs276.stanford.edu_',
 '1/cs276a.stanford.edu_',
 '3/infolab.stanford.edu_TR_CS-TN-95-21.html',
 '4/nlp.stanford.edu_IR-book_',
 '4/nlp.stanford.edu_IR-book_html_htmledition_an-appraisal-of-probabilistic-models-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_bayesian-network-approaches-to-ir-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_boolean-retrieval-2.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_computing-scores-in-a-complete-search-system-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_dictionaries-and-tolerant-retrieval-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_efficient-scoring-and-ranking-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_inexact-top-k-document-retrieval-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_probabilistic-information-retrieval-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_putting-it-all-together-1.html',
 '4/nlp.stanford.edu_IR-book_html_htmledition_references-and-further-reading-1.h

Như trước, hãy kiểm tra trên các truy vấn dev để xem nó có hoạt động đúng không

In [47]:
for i in range(1, 9):
    with open('dev_queries/query.' + str(i)) as q:
        query = q.read()
        my_results = [os.path.normpath(path) for path in BSBI_instance_compressed.retrieve(query)]
        with open('dev_output/' + str(i) + '.out') as o:
            reference_results = [os.path.normpath(x.strip()) for x in o.readlines()]
            assert my_results == reference_results, "Kết quả KHÔNG khớp cho truy vấn: "+query.strip()
        print("Kết quả khớp cho truy vấn:", query.strip())

Kết quả khớp cho truy vấn: we are
Kết quả khớp cho truy vấn: stanford class
Kết quả khớp cho truy vấn: stanford students
Kết quả khớp cho truy vấn: very cool
Kết quả khớp cho truy vấn: the
Kết quả khớp cho truy vấn: a
Kết quả khớp cho truy vấn: the the
Kết quả khớp cho truy vấn: stanford computer science


## Phân bổ điểm

Tương tự phần đánh chỉ mục không nén, bạn sẽ được kiểm tra trên 20 truy vấn kết hợp (1.5% cho mỗi truy vấn đúng, tổng cộng 30%).

Bạn sẽ bị trừ điểm nếu:
1. 15% (tổng điểm bài tập) nếu thuật toán nén không đạt chỉ mục nén có kích thước nhỏ hơn 20MB trên đĩa (chỉ file .index không bao gồm file .dict).
2. 5% (tổng điểm bài tập) nếu thời gian phản hồi truy vấn với chỉ mục nén nằm ngoài chuẩn.

Tổng điểm phần này là 30%.
*Lưu ý nếu chương trình không hiện thực thuật toán nén mã hóa độ dài biến đổi, bạn sẽ không nhận điểm cho phần này.*

## Nộp code

Bây giờ bạn sẵn sàng nộp phần code của bài tập. Tham khảo [phần hướng dẫn nộp bài](#Hướng-dẫn-nộp-bài).

### Chỉ mục nén (2%)

Hãy xem kích thước file chỉ mục nén (`.index`)

In [35]:
print("Kích thước chỉ mục nén", os.path.getsize("output_dir_compressed/BSBI.index"), 'bytes')

Kích thước chỉ mục nén 17145421 bytes


**Chỉ mục nén có nhỏ hơn chỉ mục ước tính ở phần trước không? Tại sao?
Kích thước chỉ mục nén sẽ thay đổi như thế nào nếu mã hóa byte biến đổi được áp dụng trực tiếp trên docID thay vì khoảng cách (gap)?**

> *Chỉ mục nén có kích thước nhỏ hơn ước tính*

> Lý do: do kết hợp hai kỹ thuật: Mã hóa khoảng cách (Gap Encoding) và Mã hóa byte biến đổi (Variable Byte - VB Encoding). docID tốn 1 không gian vài Byte cố định trong khi VB Encoding chỉ dùng lượng Byte vừa đủ để biểu diễn. Gap Encoding chuyển danh sách các docID lớn dần thành các chênh lệnh giữa chúng, các khoảng cách lệch này có giá trị rất nhỏ

> *Nếu mã hóa byte biến đổi được áp dụng trực tiếp trên docID thay vì khoảng cách (gap) thì :*

> Kích thước chỉ mục nén sẽ lớn hơn (tăng lên) so với khi dùng Gap. Vì trong danh sách posting, các docID được sắp xếp tăng dần và sẽ ngày càng trở nên lớn (tới hàng chục, hàng trăm ngàn) ở các vị trí sau của danh sách. Nếu áp dụng VB trực tiếp lên các docID lớn này, hệ thống sẽ cần tới 3 hoặc 4 bytes cho mỗi docID.


# Các phương pháp nén bổ sung (10% - Điểm cộng)

Hiện thực thêm một phương pháp nén chỉ mục bằng cách điền vào các phương thức `encode` và `decode` trong `ECCompressedPostings`.

Dưới đây là một số hướng:
1. [Mục 5.4](https://nlp.stanford.edu/IR-book/pdf/05comp.pdf) trong sách giáo khoa chỉ ra một số kỹ thuật (ví dụ: **gamma-encoding**) và các công bố liên quan.
2. Bạn cũng có thể tìm hiểu **Delta Encoding**, kỹ thuật nén rất giống Gamma Encoding, có thể đạt tỉ lệ nén tốt hơn.
3. Ngoài ra, [THUẬT TOÁN SIMPLE-9](https://github.com/manning/CompressionAlgorithms#simple-9) là kỹ thuật mã hóa thú vị mà bạn cũng có thể xem xét hiện thực.

Bạn nên lưu mỗi danh sách posting dưới dạng bội số của byte (thay vì bit). Các vị trí và độ dài trong file chỉ mục đều tính theo bội số của byte.

In [36]:
%%tee submission/ec_compressed_postings.py
class ECCompressedPostings:
    #Nếu bạn cần thêm phương thức trợ giúp, thêm ở đây
    ### Bắt đầu code của bạn
    import struct
    ### Kết thúc code của bạn

    @staticmethod
    def encode(postings_list):
        """Mã hóa `postings_list`

        Tham số
        ----------
        postings_list: List[int]
            Danh sách posting cần mã hóa

        Trả về
        -------
        bytes:
            Biểu diễn byte của danh sách posting đã nén
        """
        ### Bắt đầu code của bạn
        if not postings_list:
            return b''

        bit_string = ""
        last_doc = 0

        # 1. Mã hóa Gap bằng Elias Gamma
        for doc in postings_list:
            gap = doc - last_doc
            last_doc = doc

            # Gamma encoding cần giá trị > 0, ta lấy gap + 1
            val = gap + 1
            binary = bin(val)[2:] # vd: 5 -> '101'
            length_unary = '1' * (len(binary) - 1)

            # Nối unary length, số 0 ngăn cách, và phần dư nhị phân
            bit_string += length_unary + '0' + binary[1:]

        # 2. Đệm (padding) các bit '0' để độ dài chuỗi chia hết cho 8
        pad_len = (8 - len(bit_string) % 8) % 8
        bit_string += '0' * pad_len

        # 3. Chuyển đổi chuỗi bit thành bytes. Đưa số lượng postings vào 4 byte đầu để decode an toàn
        byte_arr = bytearray(struct.pack('>I', len(postings_list)))
        for i in range(0, len(bit_string), 8):
            byte_arr.append(int(bit_string[i:i+8], 2))

        return bytes(byte_arr)
        ### Kết thúc code của bạn


    @staticmethod
    def decode(encoded_postings_list):
        """Giải mã biểu diễn byte của danh sách posting đã nén

        Tham số
        ----------
        encoded_postings_list: bytes
            Biểu diễn byte được tạo bởi `CompressedPostings.encode`

        Trả về
        -------
        List[int]
            Danh sách posting đã giải mã (mỗi posting là một docId)
        """
        ### Bắt đầu code của bạn
        if not encoded_postings_list:
            return []

        # 1. Trích xuất số lượng docs từ 4 byte đầu tiên
        n_postings = struct.unpack('>I', encoded_postings_list[:4])[0]

        # 2. Biến đổi luồng byte còn lại thành chuỗi nhị phân
        bit_string = "".join(format(byte, '08b') for byte in encoded_postings_list[4:])

        postings = []
        last_doc = 0
        i = 0

        # 3. Giải mã Elias Gamma cho đúng số lượng postings
        for _ in range(n_postings):
            length = 0
            # Đọc phần unary (số lượng bit '1')
            while bit_string[i] == '1':
                length += 1
                i += 1

            i += 1 # Bỏ qua bit '0' ngăn cách

            if length == 0:
                val = 1
            else:
                offset_str = bit_string[i : i+length]
                val = (1 << length) + int(offset_str, 2)
                i += length

            gap = val - 1
            last_doc += gap
            postings.append(last_doc)

        return postings
        ### Kết thúc code của bạn

Writing submission/ec_compressed_postings.py


**Viết mô tả phương pháp nén chỉ mục của bạn cũng như thảo luận về sự đánh đổi không gian/tốc độ của phương pháp nén bổ sung này, so sánh với những gì chúng ta đã hiện thực ở các phần trước.**

> *Phương pháp được sử dụng là Elias Gamma Encoding + Gap Encoding*

>    Thay vì mã hóa trực tiếp docID, thuật toán vẫn tính khoảng cách (gap) giữa các docID liên tiếp giống như phần trước. Sau đó, mỗi giá trị gap được mã hóa theo hai phần ở cấp độ bit:

 - Phần Unary (Độ dài): Biểu diễn độ dài của chuỗi nhị phân của số đó bằng các bit 1, kết thúc bằng một bit 0.

 - Phần Binary (Giá trị): Phần nhị phân của số đó (bỏ đi bit 1 đầu tiên vì nó luôn là 1).


> *Sự đánh đổi không gian và tốc độ*

 - Về Không gian (Tối ưu hơn): Elias Gamma đạt tỷ lệ nén tốt hơn với các khoảng cách (gap) nhỏ.
 Với gap = 1 thì Elias Gamma chỉ cần 1 bit. Đối với các từ khóa phổ biến (xuất hiện dày đặc, gap rất nhỏ), Elias Gamma tiết kiệm được một lượng lớn không gian ổ đĩa và băng thông I/O khi đọc/ghi file chỉ mục.

 - Về Tốc độ (Chậm hơn): Elias Gamma chạy ở cấp độ bit. Hệ thống CPU phải liên tục thực hiện các phép toán thao tác bit phức tạp và duyệt qua từng bit một để đếm số lượng bit 1 (unary). Việc gom bit thành byte, tách byte thành bit khiến tăng thời gian xử lý của CPU.

Bạn sẽ nhận 8% cho việc trả lời đúng 20 truy vấn (0.4% mỗi truy vấn). 2% sẽ dành cho phần thảo luận không gian/tốc độ trong báo cáo.

Lưu ý vì mục tiêu của phần này là đạt tỉ lệ nén tốt hơn, bạn sẽ nhận 0% trên 10% nếu kích thước chỉ mục nén lớn hơn chỉ mục sử dụng `CompressedPostings`. Chúng tôi sẽ so sánh với hiện thực `CompressedPostings` của chúng tôi, nhưng kết quả các bài kiểm tra autograder sẽ được hiển thị để bạn biết liệu có đạt nén tốt hơn (và do đó điểm cộng) trước hạn nộp.

Bạn sẽ bị trừ điểm nếu thời gian truy xuất bất thường dài. Chúng tôi sẽ kiểm tra code để xác nhận tính đúng đắn của hiện thực nếu cần.

## Nộp điểm cộng

Bạn cần nộp lại file PDF hoàn chỉnh (bao gồm cả phần điểm cộng) đặt tên theo **mã sinh viên** tại [Google Drive nộp bài](https://drive.google.com/drive/folders/1jLkXkBEpbIa53FERm5bpN8X9yK8n9bFU?usp=sharing).